# Rules and expert systems

An **expert system** captures the knowledge of human experts as **if-then rules**, and lets a program apply them. It has three parts: a **knowledge base** of facts and rules, written with the experts; an **inference engine**, which applies the rules; and an interface that asks questions and explains its conclusions.

They were AI's first commercial success. **MYCIN** (Stanford, 1970s) recommended antibiotics for bacterial infections of the blood and meningitis, from about 600 rules, and in a blinded evaluation in 1979 its choices of therapy were rated acceptable more often than those of the Stanford specialists it was compared with; it was never used routinely on patients. **XCON** (1980) configured DEC's VAX computers from customers' orders, and was credited with saving the company tens of millions of dollars a year. The inference engine can apply the rules in two directions.

## Forward chaining

**Forward chaining** is data-driven: it starts from the known facts, fires every rule whose conditions all hold, adds the conclusions as new facts, and repeats until a pass adds nothing new. That last pass is a **fixed point**: the set of facts can't grow any further, which always happens because there are finitely many conclusions.

In [ ]:
def forward_chain(rules, facts):
    facts = set(facts)
    fired = []
    while True:
        new_facts = [conclusion for conditions, conclusion in rules
                     if conclusion not in facts and all(condition in facts for condition in conditions)]
        if not new_facts:
            return facts, fired
        facts.update(new_facts)
        fired.extend(new_facts)

A small troubleshooting system for a home internet connection. Each rule is a list of conditions and a conclusion; some conclusions are conditions of other rules, so one observation can trigger a chain.

In [ ]:
connection_rules = [
    (["router_lights_off"], "router_has_no_power"),
    (["router_lights_on", "other_devices_offline"], "internet_line_down"),
    (["router_lights_on", "other_devices_online", "this_device_offline"], "problem_in_this_device"),
    (["problem_in_this_device", "wifi_switched_off"], "advice: switch the wifi on"),
    (["internet_line_down"], "advice: call the internet provider"),
    (["router_has_no_power"], "advice: check the router's plug"),
]

observations = ["router_lights_on", "other_devices_offline"]
facts, fired = forward_chain(connection_rules, observations)
print("Derived, in order:", fired)

Derived, in order: ['internet_line_down', 'advice: call the internet provider']


Two rules fired one after the other: the observations gave `internet_line_down`, which in turn gave the advice.

## Backward chaining

**Backward chaining** is goal-driven: it starts from a hypothesis and works back. To prove a goal it looks for a rule that concludes it, and tries to prove that rule's conditions, recursively. A fact that no rule concludes has to be observed, so the system **asks the user**, and asks only the questions that matter for the current hypothesis. MYCIN worked this way, asking the physician one question at a time.

Here the user's answers come from a dictionary, so that the notebook runs on its own; a real system would call `input()`. The set `in_progress` stops a rule that depends on itself from looping forever.

In [ ]:
def backward_chain(goal, rules, ask, known=None, in_progress=None):
    known = known if known is not None else {}
    in_progress = in_progress if in_progress is not None else set()
    if goal in known:
        return known[goal]
    rules_for_goal = [conditions for conditions, conclusion in rules if conclusion == goal]
    if not rules_for_goal:
        known[goal] = ask(goal)
        return known[goal]
    in_progress.add(goal)
    proved = any(
        all(condition not in in_progress and backward_chain(condition, rules, ask, known, in_progress)
            for condition in conditions)
        for conditions in rules_for_goal
    )
    in_progress.discard(goal)
    known[goal] = proved
    return proved

In [ ]:
animal_rules = [
    (["has_feathers"], "is_bird"),
    (["has_fur", "gives_milk"], "is_mammal"),
    (["is_bird", "cannot_fly", "swims"], "is_penguin"),
    (["is_bird", "has_long_neck"], "is_ostrich"),
    (["is_mammal", "has_stripes"], "is_zebra"),
]

user_answers = {"has_feathers": True, "cannot_fly": True, "swims": True, "has_long_neck": False}
questions_asked = []


def ask_user(fact):
    questions_asked.append(fact)
    return user_answers.get(fact, False)


print("Is it a penguin?", backward_chain("is_penguin", animal_rules, ask_user))
print("Questions asked:", questions_asked)

Is it a penguin? True
Questions asked: ['has_feathers', 'cannot_fly', 'swims']


To prove `is_penguin` the system asked three questions and never asked about fur, milk or stripes: the goal decided what to ask.

## Why expert systems faded

By the late 1980s the limits were clear, and the boom ended in the second "AI winter":

- **The knowledge acquisition bottleneck.** Every rule had to be extracted from experts by hand, and experts often can't say how they decide.
- **Brittleness.** Outside the cases its rules cover, a system fails silently or absurdly: it has no common sense to fall back on.
- **Uncertainty.** Real evidence is partial. MYCIN's *certainty factors* were an ad hoc fix; probability, through **Bayesian networks**, became the principled one ([probability and Bayes' theorem](../probability/README.md)).
- **Maintenance.** Thousands of interacting rules are hard to change without breaking something.

Machine learning turned the process around: instead of writing the rules, give the program examples with their answers, and let it find the rules ([machine learning paradigms](../../background/machine_learning_paradigms.md)). Rules didn't disappear, though. **Business rule engines** decide eligibility, prices and compliance in banks and insurers; hospitals' prescribing software warns of drug interactions from rules; and the **guardrails** of a language-model application are often plain rules around a learned model ([a chatbot with guardrails](https://github.com/avidaldo/ai-chat-guardrails)).

## Cases instead of rules

Experts who can't say how they decide can still remember the cases they solved, and they often solve a new problem by recalling a similar one. **Case-based reasoning** (CBR), which grew in the early 1980s from Roger Schank's work on how memory works, keeps that experience as **cases**, problems together with the solutions that worked, and solves a new problem from the most similar ones. Agnar Aamodt and Enric Plaza (1994) described it as a cycle of four steps: **retrieve** the most similar cases, **reuse** their solution, **revise** it if it doesn't work, and **retain** the new case with its solution.

The connection problems above, as a help desk's memory of past tickets: each case is a set of symptoms and the fix that solved them.

In [ ]:
past_tickets = [
    ({"router_lights_off"}, "check the router's plug"),
    ({"router_lights_on", "other_devices_offline"}, "call the internet provider"),
    ({"router_lights_on", "other_devices_online", "this_device_offline", "wifi_switched_off"}, "switch the wifi on"),
    ({"router_lights_on", "other_devices_online", "this_device_offline", "airplane_mode_on"}, "switch airplane mode off"),
]

Retrieving needs a measure of how similar two tickets are. A simple one is the share of symptoms they have in common, out of all the symptoms either of them shows, the *Jaccard similarity*: 1 for the same symptoms, 0 for none in common.

In [ ]:
def similarity(symptoms, other_symptoms):
    return len(symptoms & other_symptoms) / len(symptoms | other_symptoms)


def retrieve(new_symptoms, cases):
    return max(cases, key=lambda case: similarity(new_symptoms, case[0]))

A user reports that the other devices are online, theirs is offline, and airplane mode is on, without saying whether the router's lights are on. The rules derive nothing from that:

In [ ]:
new_ticket = {"other_devices_online", "this_device_offline", "airplane_mode_on"}
facts, fired = forward_chain(connection_rules, new_ticket)
print("Derived by the rules:", fired)

Derived by the rules: []


Every rule about this device needs `router_lights_on`, which nobody reported, and no rule mentions airplane mode. The closest past ticket still answers:

In [ ]:
for symptoms, fix in past_tickets:
    print(f"{similarity(new_ticket, symptoms):.2f}  {fix}")
symptoms, fix = retrieve(new_ticket, past_tickets)
print("Reused fix:", fix)

0.00  check the router's plug
0.00  call the internet provider
0.40  switch the wifi on
0.75  switch airplane mode off
Reused fix: switch airplane mode off


The last ticket shares three of its four symptoms, more than any other. Where the rules are brittle, a case-based system degrades gradually: a ticket that matches no case exactly still gets the answer of the closest one. A ticket unlike every case should get none, so real systems set a minimum similarity.

Revising and retaining are how it learns. A new ticket looks like a line failure, so the fix of the closest case is to call the provider; the technician finds instead that a cable was unplugged, and the corrected case is kept:

In [ ]:
second_ticket = {"router_lights_on", "other_devices_offline", "cable_unplugged"}
symptoms, fix = retrieve(second_ticket, past_tickets)
print("Reused fix:", fix)

fix_that_worked = "plug the cable back in"
past_tickets.append((second_ticket, fix_that_worked))

Reused fix: call the internet provider


The next ticket with an unplugged cable finds the case just retained:

In [ ]:
third_ticket = {"other_devices_offline", "cable_unplugged"}
symptoms, fix = retrieve(third_ticket, past_tickets)
print("Reused fix:", fix)

Reused fix: plug the cable back in


No rule was written: the system improved by accumulating one more case. Case-based reasoning was used widely in help desks and customer support, but it has its own hard parts. The similarity measure must be designed by hand for each kind of case, and adapting a solution automatically is the hardest step, which many practical systems left to a person.

Its simplest form in machine learning is **k-nearest neighbours**, met with supervised learning: the cases are labelled rows, similarity is a distance between them, and reusing their answers is a vote or an average. The same retrieve-and-reuse pattern is how applications of language models find the stored texts most similar to a question before answering it.